In [ ]:
# 2025.11.29 LSVM Find HyperOpt & Train Result

In [1]:
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/CreditCardFraud'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [ ]:
import os

import time
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings('ignore')

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics         import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics         import roc_auc_score
from sklearn.metrics         import precision_recall_curve, classification_report
from sklearn.datasets         import make_classification

# Model import
from sklearn.tree           import DecisionTreeClassifier
from sklearn.ensemble       import RandomForestClassifier
from sklearn.ensemble       import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model   import LogisticRegression
from sklearn.linear_model   import LinearRegression
from sklearn.svm            import SVC
from sklearn.metrics        import classification_report
from xgboost                import XGBClassifier
from xgboost                import plot_importance
from lightgbm               import LGBMClassifier
from catboost               import CatBoostClassifier

# hyperopt 용
from hyperopt               import hp

# 사용자 Functions import
import importlib
from utils import user_utils
importlib.reload(user_utils)

import HyperParams          as HP 
import utils.data_sampling  as ds 

from utils import user_utils    as uu
from utils import preprocessing as pp
from utils import data_sampling as ds
from utils import model_utils   as mu
from utils import modeling      as mo

from utils.hyperopt_search import hyperopt_search, train_and_evaluate

In [3]:
# 결과받을 딕셔너리
results = {}
team_rs = 23 # 우리팀 random_state

In [4]:
#1. 데이터 로딩
raw_df = pp.ccf_load_data()

데이터 로드 성공: (284807, 31)


In [5]:
# 2. Time 컬럼 삭제 , 데이터,타겟 분리
X_features, y_target = pp.split_features_target(raw_df, cols= 'Time')

In [6]:
# 3. 이상치를 경계값으로 치환
cap_X_feature = pp.cap_outliers(X_features)

In [8]:
# 4.1 학습/테스트 데이터 분리
X_train, X_test, y_train, y_test = pp.data_split(cap_X_feature, y_target)

# 5.1 학습/검증 데이터 분리
X_tr, X_val, y_tr, y_val = pp.data_split(X_train, y_train, size=0.4)

In [10]:
# 4.2 Over Sampling 하는 경우
X_over, y_over = ds.oversampling_smote(X_train, y_train)

# 5.2 Over Sampling한 경우 학습/검증 데이터 분리
X_tr_over, X_val_over, y_tr_over, y_val_over = pp.data_split(X_over, y_over, size=0.4)

✅ SMOTE 오버샘플링 완료
   원본 샘플 수: 227845 (Class 0: 227451, Class 1: 394)
   샘플링 후: 454902 (Class 0: 227451, Class 1: 227451)


In [ ]:
# 파라미터 지정 - 에러남
# best_params, best_catboost, trials, exec_time = tuner.tune(
#     lsvc_model, X_tr, y_tr, X_val, y_val, lsvc_search_space
# )

In [ ]:
# SVM 중 LinerSVC
from sklearn.svm import LinearSVC

# Search Space 정의
lsvc_search_space = {
    'C': hp.loguniform('C', np.log(0.001), np.log(1000)),
    'class_weight': hp.choice('class_weight', [None, 'balanced']),
    'max_iter': hp.quniform('max_iter', 500, 2000, 500), # 축소, 1000, 10000, 1000),
    'tol': hp.loguniform('tol', np.log(1e-5), np.log(1e-2)),
    'dual': hp.choice('dual', [False, True]),
}

search_result = uu.hyperopt_search(
    model_class=LinearSVC,
    search_space=lsvc_search_space,
    X_train=X_tr,
    y_train=y_tr,
    max_evals=100,  
    save_trials=True,
    verbose=True
)


STEP 1: 하이퍼파라미터 탐색
  LinearSVC 하이퍼파라미터 탐색 시작
 56%|█████▌    | 56/100 [1:11:59<32:37, 44.48s/trial, best loss: -0.9856186642971867]   

In [ ]:
# 2단계: 최종 학습 및 평가
final_result = uu.train_and_evaluate(
    model_class=LinearSVC,
    params=search_result['best_params'],
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    save_model=True,
    verbose=True
)

print("\n" + "=" * 70)
print("완료!")
print("=" * 70)
print(f"최종 결과: {final_result['result_dict']}")

In [ ]:
results[model_name] = final_result['result_dict']

In [ ]:
final_result_over = train_and_evaluate(
    model_class = LinearSVC,
    params      = search_result['best_params'],
    X_train     = X_tr_over,
    y_train     = y_tr_over,
    X_test      = X_test,
    y_test      = y_test,
    save_model  = True,
    verbose     = True
)

print("\n" + "=" * 70)
print("완료!")
print("=" * 70)
print(f"최종 결과: {final_result['result_dict']}")

In [ ]:
# error version
# from sklearn.svm            import LinearSVC

# # lsvc_model = LinearSVC(random_state=team_rs)
# lsvc_search_space = {
#     'C': hp.loguniform('C', np.log(0.001), np.log(1000)),  # 정규화 강도 (작을수록 강한 정규화)
#     'class_weight': hp.choice('class_weight', [None, 'balanced']),  # 클래스 가중치
#     'max_iter': hp.quniform('max_iter', 1000, 10000, 1000),  # 최대 반복 횟수
#     'tol': hp.loguniform('tol', np.log(1e-5), np.log(1e-2)),  # 수렴 허용 오차
#     'dual': hp.choice('dual', [False, True]),  # dual formulation (n_samples > n_features일 때 False 권장)
# }

# # HyperOpt 실행
# result = hyperopt_tune(
#     model_class = LinearSVC,
#     search_space=lsvc_search_space,
#     X_train=X_train,
#     y_train=y_train,
#     X_test=X_test,
#     y_test=y_test,
# )
    
# # 결과 사용
# best_model = result['model']
# best_params = result['best_params']
# print("\n최종 결과:", result['result_dict'])
errorMsg= ''' 
==================================================
  LinearSVC 튜닝 시작
==================================================
100%|██████████| 100/100 [4:45:50<00:00, 171.51s/trial, best loss: -0.9856408454483058]  
튜닝 시간: 17150.66초
최적 roc_auc: 0.9856

---------------------------------------------------------------------------
InvalidParameterError                     Traceback (most recent call last)
Cell In[18], line 13
      4 lsvc_search_space = {
      5     'C': hp.loguniform('C', np.log(0.001), np.log(1000)),  # 정규화 강도 (작을수록 강한 정규화)
      6     'class_weight': hp.choice('class_weight', [None, 'balanced']),  # 클래스 가중치
   (...)      9     'dual': hp.choice('dual', [False, True]),  # dual formulation (n_samples > n_features일 때 False 권장)
     10 }
     12 # HyperOpt 실행
---> 13 result = hyperopt_tune(
     14     model_class = LinearSVC,
     15     search_space=lsvc_search_space,
     16     X_train=X_train,
     17     y_train=y_train,
     18     X_test=X_test,
     19     y_test=y_test,
     20 )
     22 # 결과 사용
     23 best_model = result['model']

Cell In[16], line 201, in hyperopt_tune(model_class, search_space, X_train, y_train, X_test, y_test, max_evals, cv, scoring, random_state, is_save_model, verbose)
    199 # 최종 모델 학습
    200 final_model = model_class(**final_params)
--> 201 final_model.fit(X_train, y_train)
    203 # 예측
    204 y_pred = final_model.predict(X_test)

File c:\Users\user\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\base.py:1358, in _fit_context.<locals>.decorator.<locals>.wrapper(estimator, *args, **kwargs)
   1353 partial_fit_and_fitted = (
   1354     fit_method.__name__ == "partial_fit" and _is_fitted(estimator)
   1355 )
   1357 if not global_skip_validation and not partial_fit_and_fitted:
-> 1358     estimator._validate_params()
   1360 with config_context(
   1361     skip_parameter_validation=(
   1362         prefer_skip_nested_validation or global_skip_validation
   1363     )
   1364 ):
   1365     return fit_method(estimator, *args, **kwargs)

File c:\Users\user\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\base.py:471, in BaseEstimator._validate_params(self)
    463 def _validate_params(self):
    464     """Validate types and values of constructor parameters
    465 
    466     The expected type and values must be defined in the `_parameter_constraints`
   (...)    469     accepted constraints.
    470     """
--> 471     validate_parameter_constraints(
    472         self._parameter_constraints,
    473         self.get_params(deep=False),
    474         caller_name=self.__class__.__name__,
    475     )

File c:\Users\user\anaconda3\envs\ml_dev\Lib\site-packages\sklearn\utils\_param_validation.py:98, in validate_parameter_constraints(parameter_constraints, params, caller_name)
     92 else:
     93     constraints_str = (
     94         f"{', '.join([str(c) for c in constraints[:-1]])} or"
     95         f" {constraints[-1]}"
     96     )
---> 98 raise InvalidParameterError(
     99     f"The {param_name!r} parameter of {caller_name} must be"
    100     f" {constraints_str}. Got {param_val!r} instead."
    101 )

InvalidParameterError: The 'class_weight' parameter of LinearSVC must be None, an instance of 'dict' or a str among {'balanced'}. Got 1.0 instead.
'''

In [ ]:
results['lsvc_best_opt'] = best_params

In [ ]:
# 7 시각화
mo.model_metrics_graph(results, 'XGB+LSVC 모델별 성능지표 비교')

In [ ]:
# BestOpt 찾고 나서 스케일적용 버전 만들어서 모델링하기 
xgb_best_params = {
    'colsample_bytree': 0.8,
    'gamma': 0.9657494843461217,
    'learning_rate': 0.07239930133055257,
    'max_depth': 4,
    'min_child_weight': 5,
    'n_estimators': 450,
    'reg_alpha': 0.7766700326389675,
    'reg_lambda': 8.599432523914023,
    'scale_pos_weight': 5.0,
    'subsample': 0.6000000000000001,
    "eval_metric": "auc",
    "use_label_encoder": False,
    "n_jobs": -1,
    'random_state': 23
}
# xgb_best_params_santander = {
#     "colsample_bytree": 0.88,
#     "gamma": 0.058,
#     "learning_rate": 0.13,
#     "max_depth": 6,
#     "n_estimators": 320,
#     "min_child_weight": 2,
#     "scale_pos_weight": 10,
#     "subsample": 0.85,
#     "eval_metric": "auc",
#     "use_label_encoder": False,
#     "n_jobs": -1,
#     "random_state": 23,
# }

# X_train_scaled, X_test_scaled, scaler = pp.scale_data(X_train, X_test)